In [114]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import locale
import seaborn as sns
import matplotlib.pyplot as plt
import re
import pyproj
import folium
import math

In [115]:
#importar datos de turnos de personal de control

#Turnos vía

turnos_via = pd.read_excel('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/datos_vía.xlsx')

turnos_control = pd.read_excel('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/datos_control.xlsx')

turnos_supervisor= pd.read_excel('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/datos_supervisor.xlsx')

turnos_control_2026 = pd.read_excel('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/datos_control_2026.xlsx')

turnos_motoreg = pd.read_excel('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/datos_motoregulador.xlsx')

base_turnos = pd.read_excel(
    'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/base_indicadores_fms.xlsx',
    sheet_name='lineas'
)

base_turnos

,Nombre Ruta,Id Línea,Ruta,Nombre Línea,TipoDia
0,1-1_V1,10504,10990,1-1 Alamos,Habil
1,1-1_V1,10504,10990,1-1 Alamos,Sabado
2,1-1_V1,10504,10990,1-1 Alamos,Domingo
3,C25_V3,10339,12756,C25,Habil
4,C25_V3,10339,12756,C25,Sabado
...,...,...,...,...,...
271,SE14_Vuelta_V1,10350,10668,SE14,Sabado
272,SE14_Vuelta_V1,10350,10668,SE14,Domingo
273,SE14_Vuelta_V2,10350,12739,SE14,Habil
274,SE14_Vuelta_V2,10350,12739,SE14,Sabado


In [116]:
reemplazos = {
    '16-may': '16-5 Villas Dorado',
    '5-abr': '5-4 G'
}

base_turnos['Nombre Línea '] = base_turnos['Nombre Línea '].replace(reemplazos)

base_turnos

,Nombre Ruta,Id Línea,Ruta,Nombre Línea,TipoDia
0,1-1_V1,10504,10990,1-1 Alamos,Habil
1,1-1_V1,10504,10990,1-1 Alamos,Sabado
2,1-1_V1,10504,10990,1-1 Alamos,Domingo
3,C25_V3,10339,12756,C25,Habil
4,C25_V3,10339,12756,C25,Sabado
...,...,...,...,...,...
271,SE14_Vuelta_V1,10350,10668,SE14,Sabado
272,SE14_Vuelta_V1,10350,10668,SE14,Domingo
273,SE14_Vuelta_V2,10350,12739,SE14,Habil
274,SE14_Vuelta_V2,10350,12739,SE14,Sabado


In [117]:
base_turnos_filtrado = base_turnos[
    base_turnos["Nombre Línea "] == "5-4 G"
]

base_turnos_filtrado

,Nombre Ruta,Id Línea,Ruta,Nombre Línea,TipoDia
119,5-abr,10360,10684,5-4 G,Habil
120,5-abr,10360,10684,5-4 G,Sabado
121,5-abr,10360,10684,5-4 G,Domingo


In [118]:
# Función segura para separar rutas
def separar_rutas(valor):
    if pd.isna(valor):
        return []
    
    valor = str(valor).strip()
    
    # Separar SOLO cuando haya:
    # 1. " - " (guion con espacios alrededor)
    # 2. "/" (con o sin espacios)
    rutas = re.split(r"\s*/\s*|\s-\s", valor)
    
    return [r.strip() for r in rutas if r.strip() != ""]


# Aplicar la función a la columna
turnos_via['Rutas'] = turnos_via['Rutas'].apply(separar_rutas)

# Expandir (UN REGISTRO POR RUTA)
turnos_via = turnos_via.explode('Rutas').reset_index(drop=True)

# Renombrar a Ruta
turnos_via = turnos_via.rename(columns={'Rutas':'ruta_com'})

turnos_via

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia
0,EG1,04:00:00,13:15:00,Verbena I,740,Habil
1,EG1,04:00:00,13:15:00,Verbena I,C101,Habil
2,EG1,04:00:00,13:15:00,Verbena I,DD212,Habil
3,EG2,13:15:00,22:30:00,Verbena I,740,Habil
4,EG2,13:15:00,22:30:00,Verbena I,C101,Habil
...,...,...,...,...,...,...
267,US4,14:00:00,22:00:00,Providencia,806,Domingo
268,US5,06:00:00,14:00:00,Bolonia,614,Domingo
269,US6,14:00:00,22:00:00,Bolonia,614,Domingo
270,US7,06:00:00,14:00:00,Diana Turbay,SE14,Domingo


In [119]:
# Función segura para separar rutas
def separar_rutas(valor):
    if pd.isna(valor):
        return []
    
    valor = str(valor).strip()
    
    # Separar SOLO cuando haya:
    # 1. " - " (guion con espacios alrededor)
    # 2. "/" (con o sin espacios)
    rutas = re.split(r"\s*/\s*|\s-\s", valor)
    
    return [r.strip() for r in rutas if r.strip() != ""]


# Aplicar la función a la columna
turnos_control_2026['Rutas'] = turnos_control_2026['Rutas'].apply(separar_rutas)

# Expandir (UN REGISTRO POR RUTA)
turnos_control_2026 = turnos_control_2026.explode('Rutas').reset_index(drop=True)

# Renombrar a Ruta
turnos_control_2026 = turnos_control_2026.rename(columns={'Rutas':'ruta_com'})

turnos_control_2026

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia
0,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo
1,ENLACE N,22:00:00,23:59:00,ET4,16-2 Engativa Centro,Domingo
2,ENLACE N,22:00:00,23:59:00,ET4,16-3 Av. Eldorado Alamos,Domingo
3,ENLACE N,22:00:00,23:59:00,ET4,16-4 El Muelle,Domingo
4,ENLACE N,22:00:00,23:59:00,ET4,16-5,Domingo
...,...,...,...,...,...,...
465,ET9M-V,06:00:00,14:00:00,ET9,BD237,Sabado
466,ET9T-V,14:00:00,22:00:00,ET9,C101,Sabado
467,ET9T-V,14:00:00,22:00:00,ET9,740,Sabado
468,ET9T-V,14:00:00,22:00:00,ET9,DD212,Sabado


In [120]:
# Función segura para separar rutas
def separar_rutas(valor):
    if pd.isna(valor):
        return []
    
    valor = str(valor).strip()
    
    # Separar SOLO cuando haya:
    # 1. " - " (guion con espacios alrededor)
    # 2. "/" (con o sin espacios)
    rutas = re.split(r"\s*/\s*|\s-\s", valor)
    
    return [r.strip() for r in rutas if r.strip() != ""]


# Aplicar la función a la columna
turnos_control['Rutas'] = turnos_control['Rutas'].apply(separar_rutas)

# Expandir (UN REGISTRO POR RUTA)
turnos_control = turnos_control.explode('Rutas').reset_index(drop=True)

# Renombrar a Ruta
turnos_control = turnos_control.rename(columns={'Rutas':'ruta_com'})

turnos_control

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia
0,ET6N,22:00:00,23:59:00,ET5,C25,Domingo
1,ET6N,22:00:00,23:59:00,ET5,E25,Domingo
2,ET6N,22:00:00,23:59:00,ET5,806,Domingo
3,ET6N,22:00:00,23:59:00,ET5,614,Domingo
4,ET6N,22:00:00,23:59:00,ET5,466,Domingo
...,...,...,...,...,...,...
419,ET8M,06:00:00,14:00:00,ET8,539,Habil
420,ET8M,06:00:00,14:00:00,ET8,DH216,Habil
421,ET8T,14:00:00,22:00:00,ET8,SE14,Habil
422,ET8T,14:00:00,22:00:00,ET8,539,Habil


In [121]:
# Función segura para separar rutas
def separar_rutas(valor):
    if pd.isna(valor):
        return []
    
    valor = str(valor).strip()
    
    # Separar SOLO cuando haya:
    # 1. " - " (guion con espacios alrededor)
    # 2. "/" (con o sin espacios)
    rutas = re.split(r"\s*/\s*|\s-\s", valor)
    
    return [r.strip() for r in rutas if r.strip() != ""]


# Aplicar la función a la columna
turnos_supervisor['Rutas'] = turnos_supervisor['Rutas'].apply(separar_rutas)

# Expandir (UN REGISTRO POR RUTA)
turnos_supervisor = turnos_supervisor.explode('Rutas').reset_index(drop=True)

# Renombrar a Ruta
turnos_supervisor = turnos_supervisor.rename(columns={'Rutas':'ruta_com'})

turnos_supervisor

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia
0,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo
1,ET5,22:00:00,23:59:00,SUP CCZ,E25,Domingo
2,ET5,22:00:00,23:59:00,SUP CCZ,806,Domingo
3,ET5,22:00:00,23:59:00,SUP CCZ,614,Domingo
4,ET5,22:00:00,23:59:00,SUP CCZ,466,Domingo
...,...,...,...,...,...,...
432,ET1,22:00:00,23:59:00,SUP CCZ,KL307,Sabado
433,ET1,22:00:00,23:59:00,SUP CCZ,12,Sabado
434,ET1,22:00:00,23:59:00,SUP CCZ,KB309,Sabado
435,ET1,22:00:00,23:59:00,SUP CCZ,P500,Sabado


In [122]:
# Función segura para separar rutas
def separar_rutas(valor):
    if pd.isna(valor):
        return []
    
    valor = str(valor).strip()
    
    # Separar SOLO cuando haya:
    # 1. " - " (guion con espacios alrededor)
    # 2. "/" (con o sin espacios)
    rutas = re.split(r"\s*/\s*|\s-\s", valor)
    
    return [r.strip() for r in rutas if r.strip() != ""]


# Aplicar la función a la columna
turnos_motoreg['Rutas'] = turnos_motoreg['Rutas'].apply(separar_rutas)

# Expandir (UN REGISTRO POR RUTA)
turnos_motoreg = turnos_motoreg.explode('Rutas').reset_index(drop=True)

# Renombrar a Ruta
turnos_motoreg = turnos_motoreg.rename(columns={'Rutas':'ruta_com'})

turnos_motoreg

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia
0,MR M,06:00:00,14:00:00,MR M,C25,Habil
1,MR M,06:00:00,14:00:00,MR M,E25,Habil
2,MR M,06:00:00,14:00:00,MR M,806,Habil
3,MR M,06:00:00,14:00:00,MR M,614,Habil
4,MR M,06:00:00,14:00:00,MR M,466,Habil
...,...,...,...,...,...,...
184,AUX CCZ I,08:00:00,17:00:00,AUX CCZ I,16-6 La Faena,Habil
185,AUX CCZ I,08:00:00,17:00:00,AUX CCZ I,16-14 Aeropuerto,Habil
186,AUX CCZ I,08:00:00,17:00:00,AUX CCZ I,1-1 Alamos,Habil
187,AUX CCZ I,08:00:00,17:00:00,AUX CCZ I,1-9 Villas del Dorado,Habil


In [123]:
reemplazos = {
    '1-1': '1-1 Alamos',
    '16-1': '16-1 Tierra Grata',
    '16-2': '16-2 Engativa Centro',
    '16-3': '16-3 Dorado Álamos',
    '16-4': '16-4 El Muelle',
    '16-6': '16-6 La Faena',
    '1-9': '1-9 Dorado',
    '16-14': '16-14 Aeropuerto',
    '16-5': '16-5 Villas Dorado'
}

turnos_via['ruta_com'] = turnos_via['ruta_com'].replace(reemplazos)

turnos_via

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia
0,EG1,04:00:00,13:15:00,Verbena I,740,Habil
1,EG1,04:00:00,13:15:00,Verbena I,C101,Habil
2,EG1,04:00:00,13:15:00,Verbena I,DD212,Habil
3,EG2,13:15:00,22:30:00,Verbena I,740,Habil
4,EG2,13:15:00,22:30:00,Verbena I,C101,Habil
...,...,...,...,...,...,...
267,US4,14:00:00,22:00:00,Providencia,806,Domingo
268,US5,06:00:00,14:00:00,Bolonia,614,Domingo
269,US6,14:00:00,22:00:00,Bolonia,614,Domingo
270,US7,06:00:00,14:00:00,Diana Turbay,SE14,Domingo


In [124]:
reemplazos = {
    '1-1': '1-1 Alamos',
    '16-1': '16-1 Tierra Grata',
    '16-2': '16-2 Engativa Centro',
    '16-3': '16-3 Dorado Álamos',
    '16-4': '16-4 El Muelle',
    '16-6': '16-6 La Faena',
    '1-9': '1-9 Dorado',
    '16-14': '16-14 Aeropuerto',
    '16-5': '16-5 Villas Dorado'
}

turnos_control_2026['ruta_com'] = turnos_control_2026['ruta_com'].replace(reemplazos)

turnos_control_2026

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia
0,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo
1,ENLACE N,22:00:00,23:59:00,ET4,16-2 Engativa Centro,Domingo
2,ENLACE N,22:00:00,23:59:00,ET4,16-3 Av. Eldorado Alamos,Domingo
3,ENLACE N,22:00:00,23:59:00,ET4,16-4 El Muelle,Domingo
4,ENLACE N,22:00:00,23:59:00,ET4,16-5 Villas Dorado,Domingo
...,...,...,...,...,...,...
465,ET9M-V,06:00:00,14:00:00,ET9,BD237,Sabado
466,ET9T-V,14:00:00,22:00:00,ET9,C101,Sabado
467,ET9T-V,14:00:00,22:00:00,ET9,740,Sabado
468,ET9T-V,14:00:00,22:00:00,ET9,DD212,Sabado


In [125]:
reemplazos = {
    '1-1': '1-1 Alamos',
    '16-1': '16-1 Tierra Grata',
    '16-2': '16-2 Engativa Centro',
    '16-3': '16-3 Dorado Álamos',
    '16-4': '16-4 El Muelle',
    '16-6': '16-6 La Faena',
    '1-9': '1-9 Dorado',
    '16-14': '16-14 Aeropuerto',
    '16-5': '16-5 Villas Dorado'
}

turnos_control['ruta_com'] = turnos_control['ruta_com'].replace(reemplazos)

turnos_control

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia
0,ET6N,22:00:00,23:59:00,ET5,C25,Domingo
1,ET6N,22:00:00,23:59:00,ET5,E25,Domingo
2,ET6N,22:00:00,23:59:00,ET5,806,Domingo
3,ET6N,22:00:00,23:59:00,ET5,614,Domingo
4,ET6N,22:00:00,23:59:00,ET5,466,Domingo
...,...,...,...,...,...,...
419,ET8M,06:00:00,14:00:00,ET8,539,Habil
420,ET8M,06:00:00,14:00:00,ET8,DH216,Habil
421,ET8T,14:00:00,22:00:00,ET8,SE14,Habil
422,ET8T,14:00:00,22:00:00,ET8,539,Habil


In [126]:
reemplazos = {
    '1-1': '1-1 Alamos',
    '16-1': '16-1 Tierra Grata',
    '16-2': '16-2 Engativa Centro',
    '16-3': '16-3 Dorado Álamos',
    '16-4': '16-4 El Muelle',
    '16-6': '16-6 La Faena',
    '1-9': '1-9 Dorado',
    '16-14': '16-14 Aeropuerto',
    '16-5': '16-5 Villas Dorado'
}

turnos_supervisor['ruta_com'] = turnos_supervisor['ruta_com'].replace(reemplazos)

turnos_supervisor

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia
0,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo
1,ET5,22:00:00,23:59:00,SUP CCZ,E25,Domingo
2,ET5,22:00:00,23:59:00,SUP CCZ,806,Domingo
3,ET5,22:00:00,23:59:00,SUP CCZ,614,Domingo
4,ET5,22:00:00,23:59:00,SUP CCZ,466,Domingo
...,...,...,...,...,...,...
432,ET1,22:00:00,23:59:00,SUP CCZ,KL307,Sabado
433,ET1,22:00:00,23:59:00,SUP CCZ,12,Sabado
434,ET1,22:00:00,23:59:00,SUP CCZ,KB309,Sabado
435,ET1,22:00:00,23:59:00,SUP CCZ,P500,Sabado


In [127]:
reemplazos = {
    '1-1': '1-1 Alamos',
    '16-1': '16-1 Tierra Grata',
    '16-2': '16-2 Engativa Centro',
    '16-3': '16-3 Dorado Álamos',
    '16-4': '16-4 El Muelle',
    '16-6': '16-6 La Faena',
    '1-9': '1-9 Dorado',
    '16-14': '16-14 Aeropuerto',
    '16-5': '16-5 Villas Dorado'
}

turnos_motoreg['ruta_com'] = turnos_motoreg['ruta_com'].replace(reemplazos)

turnos_motoreg

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia
0,MR M,06:00:00,14:00:00,MR M,C25,Habil
1,MR M,06:00:00,14:00:00,MR M,E25,Habil
2,MR M,06:00:00,14:00:00,MR M,806,Habil
3,MR M,06:00:00,14:00:00,MR M,614,Habil
4,MR M,06:00:00,14:00:00,MR M,466,Habil
...,...,...,...,...,...,...
184,AUX CCZ I,08:00:00,17:00:00,AUX CCZ I,16-6 La Faena,Habil
185,AUX CCZ I,08:00:00,17:00:00,AUX CCZ I,16-14 Aeropuerto,Habil
186,AUX CCZ I,08:00:00,17:00:00,AUX CCZ I,1-1 Alamos,Habil
187,AUX CCZ I,08:00:00,17:00:00,AUX CCZ I,1-9 Villas del Dorado,Habil


In [128]:
# Conteo antes
original = len(turnos_via)

# Copia de domingos
domingos = turnos_via.loc[turnos_via['TipoDia'] == 'Domingo'].copy()

# Marcar duplicado
domingos['Duplicado'] = 1
turnos_via['Duplicado'] = 0

# Concatenar
turnos_via = pd.concat([turnos_via, domingos], ignore_index=True)

# Conteo después
final = len(turnos_via)
print(f"Antes: {original}  |  Después: {final}  |  Domingos duplicados: {len(domingos)}")

turnos_via

Antes: 272  |  Después: 352  |  Domingos duplicados: 80


,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado
0,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0
1,EG1,04:00:00,13:15:00,Verbena I,C101,Habil,0
2,EG1,04:00:00,13:15:00,Verbena I,DD212,Habil,0
3,EG2,13:15:00,22:30:00,Verbena I,740,Habil,0
4,EG2,13:15:00,22:30:00,Verbena I,C101,Habil,0
...,...,...,...,...,...,...,...
347,US4,14:00:00,22:00:00,Providencia,806,Domingo,1
348,US5,06:00:00,14:00:00,Bolonia,614,Domingo,1
349,US6,14:00:00,22:00:00,Bolonia,614,Domingo,1
350,US7,06:00:00,14:00:00,Diana Turbay,SE14,Domingo,1


In [129]:
# Conteo antes
original = len(turnos_control)

# Copia de domingos
domingos = turnos_control.loc[turnos_control['TipoDia'] == 'Domingo'].copy()

# Marcar duplicado
domingos['Duplicado'] = 1
turnos_control['Duplicado'] = 0

# Concatenar
turnos_control = pd.concat([turnos_control, domingos], ignore_index=True)

# Conteo después
final = len(turnos_control)
print(f"Antes: {original}  |  Después: {final}  |  Domingos duplicados: {len(domingos)}")

turnos_control

Antes: 424  |  Después: 568  |  Domingos duplicados: 144


,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado
0,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0
1,ET6N,22:00:00,23:59:00,ET5,E25,Domingo,0
2,ET6N,22:00:00,23:59:00,ET5,806,Domingo,0
3,ET6N,22:00:00,23:59:00,ET5,614,Domingo,0
4,ET6N,22:00:00,23:59:00,ET5,466,Domingo,0
...,...,...,...,...,...,...,...
563,ET1T,14:00:00,22:00:00,ET1,12,Domingo,1
564,ET1T,14:00:00,22:00:00,ET1,DA213,Domingo,1
565,ET1T,14:00:00,22:00:00,ET1,DH209,Domingo,1
566,ET1T,14:00:00,22:00:00,ET1,DA218,Domingo,1


In [130]:
# Conteo antes
original = len(turnos_control_2026)

# Copia de domingos
domingos = turnos_control_2026.loc[turnos_control_2026['TipoDia'] == 'Domingo'].copy()

# Marcar duplicado
domingos['Duplicado'] = 1
turnos_control_2026['Duplicado'] = 0

# Concatenar
turnos_control_2026 = pd.concat([turnos_control_2026, domingos], ignore_index=True)

# Conteo después
final = len(turnos_control_2026)
print(f"Antes: {original}  |  Después: {final}  |  Domingos duplicados: {len(domingos)}")

turnos_control_2026

Antes: 470  |  Después: 606  |  Domingos duplicados: 136


,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado
0,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0
1,ENLACE N,22:00:00,23:59:00,ET4,16-2 Engativa Centro,Domingo,0
2,ENLACE N,22:00:00,23:59:00,ET4,16-3 Av. Eldorado Alamos,Domingo,0
3,ENLACE N,22:00:00,23:59:00,ET4,16-4 El Muelle,Domingo,0
4,ENLACE N,22:00:00,23:59:00,ET4,16-5 Villas Dorado,Domingo,0
...,...,...,...,...,...,...,...
601,ET8T-V,14:00:00,22:00:00,ET8,576,Domingo,1
602,ET8T-V,14:00:00,22:00:00,ET8,BD237,Domingo,1
603,ET8T-V,14:00:00,22:00:00,ET8,SE10,Domingo,1
604,ET8T-V,14:00:00,22:00:00,ET8,C101,Domingo,1


In [131]:
# Conteo antes
original = len(turnos_supervisor)

# Copia de domingos
domingos = turnos_supervisor.loc[turnos_supervisor['TipoDia'] == 'Domingo'].copy()

# Marcar duplicado
domingos['Duplicado'] = 1
turnos_supervisor['Duplicado'] = 0

# Concatenar
turnos_supervisor = pd.concat([turnos_supervisor, domingos], ignore_index=True)

# Conteo después
final = len(turnos_supervisor)
print(f"Antes: {original}  |  Después: {final}  |  Domingos duplicados: {len(domingos)}")

turnos_supervisor

Antes: 437  |  Después: 585  |  Domingos duplicados: 148


,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado
0,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0
1,ET5,22:00:00,23:59:00,SUP CCZ,E25,Domingo,0
2,ET5,22:00:00,23:59:00,SUP CCZ,806,Domingo,0
3,ET5,22:00:00,23:59:00,SUP CCZ,614,Domingo,0
4,ET5,22:00:00,23:59:00,SUP CCZ,466,Domingo,0
...,...,...,...,...,...,...,...
580,ET1,14:00:00,22:00:00,SUP CCZ,12,Domingo,1
581,ET1,14:00:00,22:00:00,SUP CCZ,DA213,Domingo,1
582,ET1,14:00:00,22:00:00,SUP CCZ,DH209,Domingo,1
583,ET1,14:00:00,22:00:00,SUP CCZ,DA218,Domingo,1


In [132]:
# Conteo antes
original = len(turnos_motoreg)

# Copia de domingos
domingos = turnos_motoreg.loc[turnos_motoreg['TipoDia'] == 'Domingo'].copy()

# Marcar duplicado
domingos['Duplicado'] = 1
turnos_motoreg['Duplicado'] = 0

# Concatenar
turnos_motoreg = pd.concat([turnos_motoreg, domingos], ignore_index=True)

# Conteo después
final = len(turnos_motoreg)
print(f"Antes: {original}  |  Después: {final}  |  Domingos duplicados: {len(domingos)}")

turnos_motoreg

Antes: 189  |  Después: 226  |  Domingos duplicados: 37


,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado
0,MR M,06:00:00,14:00:00,MR M,C25,Habil,0
1,MR M,06:00:00,14:00:00,MR M,E25,Habil,0
2,MR M,06:00:00,14:00:00,MR M,806,Habil,0
3,MR M,06:00:00,14:00:00,MR M,614,Habil,0
4,MR M,06:00:00,14:00:00,MR M,466,Habil,0
...,...,...,...,...,...,...,...
221,MR,06:00:00,14:00:00,MR,16-6 La Faena,Domingo,1
222,MR,06:00:00,14:00:00,MR,16-14 Aeropuerto,Domingo,1
223,MR,06:00:00,14:00:00,MR,1-1 Alamos,Domingo,1
224,MR,06:00:00,14:00:00,MR,1-9 Villas del Dorado,Domingo,1


In [133]:
# Asegurar limpieza básica
turnos_via['ruta_com'] = turnos_via['ruta_com'].astype(str).str.strip()
turnos_via['TipoDia'] = turnos_via['TipoDia'].astype(str).str.strip()

base_turnos['Nombre Línea '] = base_turnos['Nombre Línea '].astype(str).str.strip()
base_turnos['TipoDia'] = base_turnos['TipoDia'].astype(str).str.strip()

# Seleccionar solo lo necesario desde base_turnos
base_ref = base_turnos[['Nombre Línea ','TipoDia','Id Línea ','Ruta ']].copy()
base_ref = base_ref.rename(columns={'Ruta ':'Ruta_com'})

# Hacer el merge
turnos_via = turnos_via.merge(
    base_ref,
    left_on=['ruta_com','TipoDia'],
    right_on=['Nombre Línea ','TipoDia'],
    how='left'
)

# Eliminar la columna auxiliar
turnos_via.drop(columns=['Nombre Línea '], inplace=True)

turnos_via

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado,Id Línea,Ruta_com
0,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415
1,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,12774
2,EG1,04:00:00,13:15:00,Verbena I,C101,Habil,0,10310,12271
3,EG1,04:00:00,13:15:00,Verbena I,C101,Habil,0,10310,12271
4,EG1,04:00:00,13:15:00,Verbena I,C101,Habil,0,10310,12777
...,...,...,...,...,...,...,...,...,...
937,US7,06:00:00,14:00:00,Diana Turbay,SE14,Domingo,1,10350,12739
938,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,12251
939,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,12738
940,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,10668


In [134]:
# Asegurar limpieza básica
turnos_control['ruta_com'] = turnos_control['ruta_com'].astype(str).str.strip()
turnos_control['TipoDia'] = turnos_control['TipoDia'].astype(str).str.strip()

base_turnos['Nombre Línea '] = base_turnos['Nombre Línea '].astype(str).str.strip()
base_turnos['TipoDia'] = base_turnos['TipoDia'].astype(str).str.strip()

# Seleccionar solo lo necesario desde base_turnos
base_ref = base_turnos[['Nombre Línea ','TipoDia','Id Línea ','Ruta ']].copy()
base_ref = base_ref.rename(columns={'Ruta ':'Ruta_com'})

# Hacer el merge
turnos_control = turnos_control.merge(
    base_ref,
    left_on=['ruta_com','TipoDia'],
    right_on=['Nombre Línea ','TipoDia'],
    how='left'
)

# Eliminar la columna auxiliar
turnos_control.drop(columns=['Nombre Línea '], inplace=True)

turnos_control

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado,Id Línea,Ruta_com
0,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12756.0
1,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12607.0
2,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12757.0
3,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12606.0
4,ET6N,22:00:00,23:59:00,ET5,E25,Domingo,0,10196.0,12784.0
...,...,...,...,...,...,...,...,...,...
1453,ET1T,14:00:00,22:00:00,ET1,DA218,Domingo,1,10691.0,12746.0
1454,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12373.0
1455,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12364.0
1456,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12721.0


In [135]:
# Asegurar limpieza básica
turnos_control_2026['ruta_com'] = turnos_control_2026['ruta_com'].astype(str).str.strip()
turnos_control_2026['TipoDia'] = turnos_control_2026['TipoDia'].astype(str).str.strip()

base_turnos['Nombre Línea '] = base_turnos['Nombre Línea '].astype(str).str.strip()
base_turnos['TipoDia'] = base_turnos['TipoDia'].astype(str).str.strip()

# Seleccionar solo lo necesario desde base_turnos
base_ref = base_turnos[['Nombre Línea ','TipoDia','Id Línea ','Ruta ']].copy()
base_ref = base_ref.rename(columns={'Ruta ':'Ruta_com'})

# Hacer el merge
turnos_control_2026 = turnos_control_2026.merge(
    base_ref,
    left_on=['ruta_com','TipoDia'],
    right_on=['Nombre Línea ','TipoDia'],
    how='left'
)

# Eliminar la columna auxiliar
turnos_control_2026.drop(columns=['Nombre Línea '], inplace=True)

turnos_control_2026

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado,Id Línea,Ruta_com
0,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0,10492,10963
1,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0,10492,10960
2,ENLACE N,22:00:00,23:59:00,ET4,16-2 Engativa Centro,Domingo,0,10474,12552
3,ENLACE N,22:00:00,23:59:00,ET4,16-2 Engativa Centro,Domingo,0,10474,12551
4,ENLACE N,22:00:00,23:59:00,ET4,16-3 Av. Eldorado Alamos,Domingo,0,10478,10941
...,...,...,...,...,...,...,...,...,...
1554,ET8T-V,14:00:00,22:00:00,ET8,C101,Domingo,1,10310,12272
1555,ET8T-V,14:00:00,22:00:00,ET8,C101,Domingo,1,10310,12272
1556,ET8T-V,14:00:00,22:00:00,ET8,C101,Domingo,1,10310,12778
1557,ET8T-V,14:00:00,22:00:00,ET8,740,Domingo,1,10184,10415


In [136]:
# Asegurar limpieza básica
turnos_supervisor['ruta_com'] = turnos_supervisor['ruta_com'].astype(str).str.strip()
turnos_supervisor['TipoDia'] = turnos_supervisor['TipoDia'].astype(str).str.strip()

base_turnos['Nombre Línea '] = base_turnos['Nombre Línea '].astype(str).str.strip()
base_turnos['TipoDia'] = base_turnos['TipoDia'].astype(str).str.strip()

# Seleccionar solo lo necesario desde base_turnos
base_ref = base_turnos[['Nombre Línea ','TipoDia','Id Línea ','Ruta ']].copy()
base_ref = base_ref.rename(columns={'Ruta ':'Ruta_com'})

# Hacer el merge
turnos_supervisor = turnos_supervisor.merge(
    base_ref,
    left_on=['ruta_com','TipoDia'],
    right_on=['Nombre Línea ','TipoDia'],
    how='left'
)

# Eliminar la columna auxiliar
turnos_supervisor.drop(columns=['Nombre Línea '], inplace=True)

turnos_supervisor

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado,Id Línea,Ruta_com
0,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12756
1,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12607
2,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12757
3,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12606
4,ET5,22:00:00,23:59:00,SUP CCZ,E25,Domingo,0,10196,12784
...,...,...,...,...,...,...,...,...,...
1487,ET1,14:00:00,22:00:00,SUP CCZ,DA218,Domingo,1,10691,12746
1488,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12373
1489,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12364
1490,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12721


In [137]:
# Asegurar limpieza básica
turnos_motoreg['ruta_com'] = turnos_motoreg['ruta_com'].astype(str).str.strip()
turnos_motoreg['TipoDia'] = turnos_motoreg['TipoDia'].astype(str).str.strip()

base_turnos['Nombre Línea '] = base_turnos['Nombre Línea '].astype(str).str.strip()
base_turnos['TipoDia'] = base_turnos['TipoDia'].astype(str).str.strip()

# Seleccionar solo lo necesario desde base_turnos
base_ref = base_turnos[['Nombre Línea ','TipoDia','Id Línea ','Ruta ']].copy()
base_ref = base_ref.rename(columns={'Ruta ':'Ruta_com'})

# Hacer el merge
turnos_motoreg = turnos_motoreg.merge(
    base_ref,
    left_on=['ruta_com','TipoDia'],
    right_on=['Nombre Línea ','TipoDia'],
    how='left'
)

# Eliminar la columna auxiliar
turnos_motoreg.drop(columns=['Nombre Línea '], inplace=True)

turnos_motoreg

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado,Id Línea,Ruta_com
0,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756
1,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12607
2,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12757
3,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12606
4,MR M,06:00:00,14:00:00,MR M,E25,Habil,0,10196,12784
...,...,...,...,...,...,...,...,...,...
573,MR,06:00:00,14:00:00,MR,16-14 Aeropuerto,Domingo,1,10495,10965
574,MR,06:00:00,14:00:00,MR,16-14 Aeropuerto,Domingo,1,10495,10971
575,MR,06:00:00,14:00:00,MR,1-1 Alamos,Domingo,1,10504,10990
576,MR,06:00:00,14:00:00,MR,1-9 Villas del Dorado,Domingo,1,10527,12440


In [138]:
#Duplicar columnas
turnos_via['Inicio Funciones TP20'] = turnos_via['Inicio Funciones (TP20)']
turnos_via['Fin Funciones TP21'] = turnos_via['Fin Funciones (TP21)']

In [139]:
#Duplicar columnas
turnos_control['Inicio Funciones TP20'] = turnos_control['Inicio Funciones (TP20)']
turnos_control['Fin Funciones TP21'] = turnos_control['Fin Funciones (TP21)']

In [140]:
#Duplicar columnas
turnos_control_2026['Inicio Funciones TP20'] = turnos_control_2026['Inicio Funciones (TP20)']
turnos_control_2026['Fin Funciones TP21'] = turnos_control_2026['Fin Funciones (TP21)']

In [141]:
#Duplicar columnas
turnos_supervisor['Inicio Funciones TP20'] = turnos_supervisor['Inicio Funciones (TP20)']
turnos_supervisor['Fin Funciones TP21'] = turnos_supervisor['Fin Funciones (TP21)']

In [142]:
#Duplicar columnas
turnos_motoreg['Inicio Funciones TP20'] = turnos_motoreg['Inicio Funciones (TP20)']
turnos_motoreg['Fin Funciones TP21'] = turnos_motoreg['Fin Funciones (TP21)']

In [143]:
def parsear_hms_a_minutos(hms):
    if pd.isna(hms):
        return None
    
    s = str(hms).strip()

    # Acepta formatos: H:MM , HH:MM , HH:MM:SS
    m = re.match(r"^(\d{1,3}):(\d{2})(?::(\d{2}))?$", s)
    
    if not m:
        return None
    
    h = int(m.group(1))
    mm = int(m.group(2))
    ss = int(m.group(3)) if m.group(3) else 0

    return (h * 60) + mm


# =========================================
# 2. Generar lista de horas (franjas) por fila
# =========================================
def generar_horas_por_fila(fila):

    inicio_str = fila['Inicio Funciones (TP20)']
    fin_str    = fila['Fin Funciones (TP21)']

    inicio_min = parsear_hms_a_minutos(inicio_str)
    fin_min    = parsear_hms_a_minutos(fin_str)

    if inicio_min is None or fin_min is None:
        return []

    inicio_h = inicio_min // 60
    fin_h    = fin_min // 60

    if fin_h < inicio_h:
        return []

    # Generamos todas las horas
    return list(range(int(inicio_h), int(fin_h) + 1))


In [144]:
def parsear_hms_a_minutos(hms):
    if pd.isna(hms):
        return None
    
    s = str(hms).strip()

    # Acepta formatos: H:MM , HH:MM , HH:MM:SS
    m = re.match(r"^(\d{1,3}):(\d{2})(?::(\d{2}))?$", s)
    
    if not m:
        return None
    
    h = int(m.group(1))
    mm = int(m.group(2))
    ss = int(m.group(3)) if m.group(3) else 0

    return (h * 60) + mm


# =========================================
# 2. Generar lista de horas (franjas) por fila
# =========================================
def generar_horas_por_fila(fila):

    inicio_str = fila['Inicio Funciones (TP20)']
    fin_str    = fila['Fin Funciones (TP21)']

    inicio_min = parsear_hms_a_minutos(inicio_str)
    fin_min    = parsear_hms_a_minutos(fin_str)

    if inicio_min is None or fin_min is None:
        return []

    inicio_h = inicio_min // 60
    fin_h    = fin_min // 60

    if fin_h < inicio_h:
        return []

    # Generamos todas las horas
    return list(range(int(inicio_h), int(fin_h) + 1))

In [145]:
def parsear_hms_a_minutos(hms):
    if pd.isna(hms):
        return None
    
    s = str(hms).strip()

    # Acepta formatos: H:MM , HH:MM , HH:MM:SS
    m = re.match(r"^(\d{1,3}):(\d{2})(?::(\d{2}))?$", s)
    
    if not m:
        return None
    
    h = int(m.group(1))
    mm = int(m.group(2))
    ss = int(m.group(3)) if m.group(3) else 0

    return (h * 60) + mm


# =========================================
# 2. Generar lista de horas (franjas) por fila
# =========================================
def generar_horas_por_fila(fila):

    inicio_str = fila['Inicio Funciones (TP20)']
    fin_str    = fila['Fin Funciones (TP21)']

    inicio_min = parsear_hms_a_minutos(inicio_str)
    fin_min    = parsear_hms_a_minutos(fin_str)

    if inicio_min is None or fin_min is None:
        return []

    inicio_h = inicio_min // 60
    fin_h    = fin_min // 60

    if fin_h < inicio_h:
        return []

    # Generamos todas las horas
    return list(range(int(inicio_h), int(fin_h) + 1))

In [146]:
turnos_via['horas'] = turnos_via.apply(generar_horas_por_fila, axis=1)

# Expandir registros
turnos_via = turnos_via.explode('horas').reset_index(drop=True)

# Crear Franja
turnos_via['Franja'] = turnos_via['horas']

# Si quieres también la hora tipo 00:00:00
turnos_via['Hora_str'] = turnos_via['horas'].astype(str) + ':00:00'

turnos_via

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado,Id Línea,Ruta_com,Inicio Funciones TP20,Fin Funciones TP21,horas,Franja,Hora_str
0,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415,04:00:00,13:15:00,4,4,4:00:00
1,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415,04:00:00,13:15:00,5,5,5:00:00
2,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415,04:00:00,13:15:00,6,6,6:00:00
3,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415,04:00:00,13:15:00,7,7,7:00:00
4,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415,04:00:00,13:15:00,8,8,8:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8745,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,12739,14:00:00,22:00:00,18,18,18:00:00
8746,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,12739,14:00:00,22:00:00,19,19,19:00:00
8747,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,12739,14:00:00,22:00:00,20,20,20:00:00
8748,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,12739,14:00:00,22:00:00,21,21,21:00:00


In [147]:
turnos_control['horas'] = turnos_control.apply(generar_horas_por_fila, axis=1)

# Expandir registros
turnos_control = turnos_control.explode('horas').reset_index(drop=True)

# Crear Franja
turnos_control['Franja'] = turnos_control['horas']

# Si quieres también la hora tipo 00:00:00
turnos_control['Hora_str'] = turnos_control['horas'].astype(str) + ':00:00'

turnos_control

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado,Id Línea,Ruta_com,Inicio Funciones TP20,Fin Funciones TP21,horas,Franja,Hora_str
0,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12756.0,22:00:00,23:59:00,22,22,22:00:00
1,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12756.0,22:00:00,23:59:00,23,23,23:00:00
2,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12607.0,22:00:00,23:59:00,22,22,22:00:00
3,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12607.0,22:00:00,23:59:00,23,23,23:00:00
4,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12757.0,22:00:00,23:59:00,22,22,22:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9634,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12722.0,14:00:00,22:00:00,18,18,18:00:00
9635,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12722.0,14:00:00,22:00:00,19,19,19:00:00
9636,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12722.0,14:00:00,22:00:00,20,20,20:00:00
9637,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12722.0,14:00:00,22:00:00,21,21,21:00:00


In [148]:
turnos_control_2026['horas'] = turnos_control_2026.apply(generar_horas_por_fila, axis=1)

# Expandir registros
turnos_control_2026 = turnos_control_2026.explode('horas').reset_index(drop=True)

# Crear Franja
turnos_control_2026['Franja'] = turnos_control_2026['horas']

# Si quieres también la hora tipo 00:00:00
turnos_control_2026['Hora_str'] = turnos_control_2026['horas'].astype(str) + ':00:00'

turnos_control_2026

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado,Id Línea,Ruta_com,Inicio Funciones TP20,Fin Funciones TP21,horas,Franja,Hora_str
0,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0,10492,10963,22:00:00,23:59:00,22,22,22:00:00
1,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0,10492,10963,22:00:00,23:59:00,23,23,23:00:00
2,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0,10492,10960,22:00:00,23:59:00,22,22,22:00:00
3,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0,10492,10960,22:00:00,23:59:00,23,23,23:00:00
4,ENLACE N,22:00:00,23:59:00,ET4,16-2 Engativa Centro,Domingo,0,10474,12552,22:00:00,23:59:00,22,22,22:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,ET8T-V,14:00:00,22:00:00,ET8,740,Domingo,1,10184,12774,14:00:00,22:00:00,18,18,18:00:00
9996,ET8T-V,14:00:00,22:00:00,ET8,740,Domingo,1,10184,12774,14:00:00,22:00:00,19,19,19:00:00
9997,ET8T-V,14:00:00,22:00:00,ET8,740,Domingo,1,10184,12774,14:00:00,22:00:00,20,20,20:00:00
9998,ET8T-V,14:00:00,22:00:00,ET8,740,Domingo,1,10184,12774,14:00:00,22:00:00,21,21,21:00:00


In [149]:
turnos_supervisor['horas'] = turnos_supervisor.apply(generar_horas_por_fila, axis=1)

# Expandir registros
turnos_supervisor = turnos_supervisor.explode('horas').reset_index(drop=True)

# Crear Franja
turnos_supervisor['Franja'] = turnos_supervisor['horas']

# Si quieres también la hora tipo 00:00:00
turnos_supervisor['Hora_str'] = turnos_supervisor['horas'].astype(str) + ':00:00'

turnos_supervisor

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado,Id Línea,Ruta_com,Inicio Funciones TP20,Fin Funciones TP21,horas,Franja,Hora_str
0,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12756,22:00:00,23:59:00,22,22,22:00:00
1,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12756,22:00:00,23:59:00,23,23,23:00:00
2,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12607,22:00:00,23:59:00,22,22,22:00:00
3,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12607,22:00:00,23:59:00,23,23,23:00:00
4,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12757,22:00:00,23:59:00,22,22,22:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9850,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12722,14:00:00,22:00:00,18,18,18:00:00
9851,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12722,14:00:00,22:00:00,19,19,19:00:00
9852,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12722,14:00:00,22:00:00,20,20,20:00:00
9853,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12722,14:00:00,22:00:00,21,21,21:00:00


In [150]:
turnos_motoreg['horas'] = turnos_motoreg.apply(generar_horas_por_fila, axis=1)

# Expandir registros
turnos_motoreg = turnos_motoreg.explode('horas').reset_index(drop=True)

# Crear Franja
turnos_motoreg['Franja'] = turnos_motoreg['horas']

# Si quieres también la hora tipo 00:00:00
turnos_motoreg['Hora_str'] = turnos_motoreg['horas'].astype(str) + ':00:00'

turnos_motoreg

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Sitio Funciones,ruta_com,TipoDia,Duplicado,Id Línea,Ruta_com,Inicio Funciones TP20,Fin Funciones TP21,horas,Franja,Hora_str
0,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756,06:00:00,14:00:00,6,6,6:00:00
1,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756,06:00:00,14:00:00,7,7,7:00:00
2,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756,06:00:00,14:00:00,8,8,8:00:00
3,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756,06:00:00,14:00:00,9,9,9:00:00
4,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756,06:00:00,14:00:00,10,10,10:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5379,MR,06:00:00,14:00:00,MR,5-4 G,Domingo,1,10360,10684,06:00:00,14:00:00,10,10,10:00:00
5380,MR,06:00:00,14:00:00,MR,5-4 G,Domingo,1,10360,10684,06:00:00,14:00:00,11,11,11:00:00
5381,MR,06:00:00,14:00:00,MR,5-4 G,Domingo,1,10360,10684,06:00:00,14:00:00,12,12,12:00:00
5382,MR,06:00:00,14:00:00,MR,5-4 G,Domingo,1,10360,10684,06:00:00,14:00:00,13,13,13:00:00


In [151]:
# Renombrar columnas
turnos_via = turnos_via.rename(columns={'TipoDia':'Tipo Dia',
                                        'ruta_com':'Ruta',
                                        'Id Línea ':'Linea',
                                        'Ruta_com':'Ruta_SAE',
                                        'Sitio Funciones':'Estacion'})

turnos_via

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Estacion,Ruta,Tipo Dia,Duplicado,Linea,Ruta_SAE,Inicio Funciones TP20,Fin Funciones TP21,horas,Franja,Hora_str
0,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415,04:00:00,13:15:00,4,4,4:00:00
1,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415,04:00:00,13:15:00,5,5,5:00:00
2,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415,04:00:00,13:15:00,6,6,6:00:00
3,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415,04:00:00,13:15:00,7,7,7:00:00
4,EG1,04:00:00,13:15:00,Verbena I,740,Habil,0,10184,10415,04:00:00,13:15:00,8,8,8:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8745,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,12739,14:00:00,22:00:00,18,18,18:00:00
8746,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,12739,14:00:00,22:00:00,19,19,19:00:00
8747,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,12739,14:00:00,22:00:00,20,20,20:00:00
8748,US8,14:00:00,22:00:00,Diana Turbay,SE14,Domingo,1,10350,12739,14:00:00,22:00:00,21,21,21:00:00


In [152]:
# Renombrar columnas
turnos_control = turnos_control.rename(columns={'TipoDia':'Tipo Dia',
                                        'ruta_com':'Ruta',
                                        'Id Línea ':'Linea',
                                        'Ruta_com':'Ruta_SAE',
                                        'Sitio Funciones':'Estacion'})

turnos_control

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Estacion,Ruta,Tipo Dia,Duplicado,Linea,Ruta_SAE,Inicio Funciones TP20,Fin Funciones TP21,horas,Franja,Hora_str
0,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12756.0,22:00:00,23:59:00,22,22,22:00:00
1,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12756.0,22:00:00,23:59:00,23,23,23:00:00
2,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12607.0,22:00:00,23:59:00,22,22,22:00:00
3,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12607.0,22:00:00,23:59:00,23,23,23:00:00
4,ET6N,22:00:00,23:59:00,ET5,C25,Domingo,0,10339.0,12757.0,22:00:00,23:59:00,22,22,22:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9634,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12722.0,14:00:00,22:00:00,18,18,18:00:00
9635,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12722.0,14:00:00,22:00:00,19,19,19:00:00
9636,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12722.0,14:00:00,22:00:00,20,20,20:00:00
9637,ET1T,14:00:00,22:00:00,ET1,DL219,Domingo,1,10690.0,12722.0,14:00:00,22:00:00,21,21,21:00:00


In [153]:
# Renombrar columnas
turnos_control_2026 = turnos_control_2026.rename(columns={'TipoDia':'Tipo Dia',
                                        'ruta_com':'Ruta',
                                        'Id Línea ':'Linea',
                                        'Ruta_com':'Ruta_SAE',
                                        'Sitio Funciones':'Estacion'})

turnos_control_2026

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Estacion,Ruta,Tipo Dia,Duplicado,Linea,Ruta_SAE,Inicio Funciones TP20,Fin Funciones TP21,horas,Franja,Hora_str
0,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0,10492,10963,22:00:00,23:59:00,22,22,22:00:00
1,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0,10492,10963,22:00:00,23:59:00,23,23,23:00:00
2,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0,10492,10960,22:00:00,23:59:00,22,22,22:00:00
3,ENLACE N,22:00:00,23:59:00,ET4,16-1 Tierra Grata,Domingo,0,10492,10960,22:00:00,23:59:00,23,23,23:00:00
4,ENLACE N,22:00:00,23:59:00,ET4,16-2 Engativa Centro,Domingo,0,10474,12552,22:00:00,23:59:00,22,22,22:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,ET8T-V,14:00:00,22:00:00,ET8,740,Domingo,1,10184,12774,14:00:00,22:00:00,18,18,18:00:00
9996,ET8T-V,14:00:00,22:00:00,ET8,740,Domingo,1,10184,12774,14:00:00,22:00:00,19,19,19:00:00
9997,ET8T-V,14:00:00,22:00:00,ET8,740,Domingo,1,10184,12774,14:00:00,22:00:00,20,20,20:00:00
9998,ET8T-V,14:00:00,22:00:00,ET8,740,Domingo,1,10184,12774,14:00:00,22:00:00,21,21,21:00:00


In [154]:
# Renombrar columnas
turnos_supervisor = turnos_supervisor.rename(columns={'TipoDia':'Tipo Dia',
                                        'ruta_com':'Ruta',
                                        'Id Línea ':'Linea',
                                        'Ruta_com':'Ruta_SAE',
                                        'Sitio Funciones':'Estacion'})

turnos_supervisor

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Estacion,Ruta,Tipo Dia,Duplicado,Linea,Ruta_SAE,Inicio Funciones TP20,Fin Funciones TP21,horas,Franja,Hora_str
0,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12756,22:00:00,23:59:00,22,22,22:00:00
1,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12756,22:00:00,23:59:00,23,23,23:00:00
2,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12607,22:00:00,23:59:00,22,22,22:00:00
3,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12607,22:00:00,23:59:00,23,23,23:00:00
4,ET5,22:00:00,23:59:00,SUP CCZ,C25,Domingo,0,10339,12757,22:00:00,23:59:00,22,22,22:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9850,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12722,14:00:00,22:00:00,18,18,18:00:00
9851,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12722,14:00:00,22:00:00,19,19,19:00:00
9852,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12722,14:00:00,22:00:00,20,20,20:00:00
9853,ET1,14:00:00,22:00:00,SUP CCZ,DL219,Domingo,1,10690,12722,14:00:00,22:00:00,21,21,21:00:00


In [155]:
# Renombrar columnas
turnos_motoreg = turnos_motoreg.rename(columns={'TipoDia':'Tipo Dia',
                                        'ruta_com':'Ruta',
                                        'Id Línea ':'Linea',
                                        'Ruta_com':'Ruta_SAE',
                                        'Sitio Funciones':'Estacion'})

turnos_motoreg

,Turno,Inicio Funciones (TP20),Fin Funciones (TP21),Estacion,Ruta,Tipo Dia,Duplicado,Linea,Ruta_SAE,Inicio Funciones TP20,Fin Funciones TP21,horas,Franja,Hora_str
0,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756,06:00:00,14:00:00,6,6,6:00:00
1,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756,06:00:00,14:00:00,7,7,7:00:00
2,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756,06:00:00,14:00:00,8,8,8:00:00
3,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756,06:00:00,14:00:00,9,9,9:00:00
4,MR M,06:00:00,14:00:00,MR M,C25,Habil,0,10339,12756,06:00:00,14:00:00,10,10,10:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5379,MR,06:00:00,14:00:00,MR,5-4 G,Domingo,1,10360,10684,06:00:00,14:00:00,10,10,10:00:00
5380,MR,06:00:00,14:00:00,MR,5-4 G,Domingo,1,10360,10684,06:00:00,14:00:00,11,11,11:00:00
5381,MR,06:00:00,14:00:00,MR,5-4 G,Domingo,1,10360,10684,06:00:00,14:00:00,12,12,12:00:00
5382,MR,06:00:00,14:00:00,MR,5-4 G,Domingo,1,10360,10684,06:00:00,14:00:00,13,13,13:00:00


In [156]:
columnas = [
    'Tipo Dia',
    'Ruta',
    'Linea',
    'Ruta_SAE',
    'Estacion',
    'Franja',
    'Turno'
]

turnos_via = turnos_via.loc[:, columnas]

turnos_via['Tipo Dia'] = (
    turnos_via['Tipo Dia']
    .astype(str)
    .str.strip()
    .str[0]   # solo la primera letra
    .str.upper()
)

reemplazos = {
    'D': 'F'
}

turnos_via['Tipo Dia'] = turnos_via['Tipo Dia'].replace(reemplazos)

turnos_via

,Tipo Dia,Ruta,Linea,Ruta_SAE,Estacion,Franja,Turno
0,H,740,10184,10415,Verbena I,4,EG1
1,H,740,10184,10415,Verbena I,5,EG1
2,H,740,10184,10415,Verbena I,6,EG1
3,H,740,10184,10415,Verbena I,7,EG1
4,H,740,10184,10415,Verbena I,8,EG1
...,...,...,...,...,...,...,...
8745,F,SE14,10350,12739,Diana Turbay,18,US8
8746,F,SE14,10350,12739,Diana Turbay,19,US8
8747,F,SE14,10350,12739,Diana Turbay,20,US8
8748,F,SE14,10350,12739,Diana Turbay,21,US8


In [157]:
columnas = [
    'Tipo Dia',
    'Ruta',
    'Linea',
    'Ruta_SAE',
    'Estacion',
    'Franja',
    'Turno'
]

turnos_control = turnos_control.loc[:, columnas]

turnos_control['Tipo Dia'] = (
    turnos_control['Tipo Dia']
    .astype(str)
    .str.strip()
    .str[0]   # solo la primera letra
    .str.upper()
)

reemplazos = {
    'D': 'F'
}

turnos_control['Tipo Dia'] = turnos_control['Tipo Dia'].replace(reemplazos)

turnos_control

,Tipo Dia,Ruta,Linea,Ruta_SAE,Estacion,Franja,Turno
0,F,C25,10339.0,12756.0,ET5,22,ET6N
1,F,C25,10339.0,12756.0,ET5,23,ET6N
2,F,C25,10339.0,12607.0,ET5,22,ET6N
3,F,C25,10339.0,12607.0,ET5,23,ET6N
4,F,C25,10339.0,12757.0,ET5,22,ET6N
...,...,...,...,...,...,...,...
9634,F,DL219,10690.0,12722.0,ET1,18,ET1T
9635,F,DL219,10690.0,12722.0,ET1,19,ET1T
9636,F,DL219,10690.0,12722.0,ET1,20,ET1T
9637,F,DL219,10690.0,12722.0,ET1,21,ET1T


In [158]:
columnas = [
    'Tipo Dia',
    'Ruta',
    'Linea',
    'Ruta_SAE',
    'Estacion',
    'Franja',
    'Turno'
]

turnos_control_2026 = turnos_control_2026.loc[:, columnas]

turnos_control_2026['Tipo Dia'] = (
    turnos_control_2026['Tipo Dia']
    .astype(str)
    .str.strip()
    .str[0]   # solo la primera letra
    .str.upper()
)

reemplazos = {
    'D': 'F'
}

turnos_control_2026['Tipo Dia'] = turnos_control_2026['Tipo Dia'].replace(reemplazos)

turnos_control_2026

,Tipo Dia,Ruta,Linea,Ruta_SAE,Estacion,Franja,Turno
0,F,16-1 Tierra Grata,10492,10963,ET4,22,ENLACE N
1,F,16-1 Tierra Grata,10492,10963,ET4,23,ENLACE N
2,F,16-1 Tierra Grata,10492,10960,ET4,22,ENLACE N
3,F,16-1 Tierra Grata,10492,10960,ET4,23,ENLACE N
4,F,16-2 Engativa Centro,10474,12552,ET4,22,ENLACE N
...,...,...,...,...,...,...,...
9995,F,740,10184,12774,ET8,18,ET8T-V
9996,F,740,10184,12774,ET8,19,ET8T-V
9997,F,740,10184,12774,ET8,20,ET8T-V
9998,F,740,10184,12774,ET8,21,ET8T-V


In [159]:
columnas = [
    'Tipo Dia',
    'Ruta',
    'Linea',
    'Ruta_SAE',
    'Estacion',
    'Franja',
    'Turno'
]

turnos_supervisor = turnos_supervisor.loc[:, columnas]

turnos_supervisor['Tipo Dia'] = (
    turnos_supervisor['Tipo Dia']
    .astype(str)
    .str.strip()
    .str[0]   # solo la primera letra
    .str.upper()
)

reemplazos = {
    'D': 'F'
}

turnos_supervisor['Tipo Dia'] = turnos_supervisor['Tipo Dia'].replace(reemplazos)

turnos_supervisor

,Tipo Dia,Ruta,Linea,Ruta_SAE,Estacion,Franja,Turno
0,F,C25,10339,12756,SUP CCZ,22,ET5
1,F,C25,10339,12756,SUP CCZ,23,ET5
2,F,C25,10339,12607,SUP CCZ,22,ET5
3,F,C25,10339,12607,SUP CCZ,23,ET5
4,F,C25,10339,12757,SUP CCZ,22,ET5
...,...,...,...,...,...,...,...
9850,F,DL219,10690,12722,SUP CCZ,18,ET1
9851,F,DL219,10690,12722,SUP CCZ,19,ET1
9852,F,DL219,10690,12722,SUP CCZ,20,ET1
9853,F,DL219,10690,12722,SUP CCZ,21,ET1


In [160]:
columnas = [
    'Tipo Dia',
    'Ruta',
    'Linea',
    'Ruta_SAE',
    'Estacion',
    'Franja',
    'Turno'
]

turnos_motoreg= turnos_motoreg.loc[:, columnas]

turnos_motoreg['Tipo Dia'] = (
    turnos_motoreg['Tipo Dia']
    .astype(str)
    .str.strip()
    .str[0]   # solo la primera letra
    .str.upper()
)

reemplazos = {
    'D': 'F'
}

turnos_motoreg['Tipo Dia'] = turnos_motoreg['Tipo Dia'].replace(reemplazos)

turnos_motoreg

,Tipo Dia,Ruta,Linea,Ruta_SAE,Estacion,Franja,Turno
0,H,C25,10339,12756,MR M,6,MR M
1,H,C25,10339,12756,MR M,7,MR M
2,H,C25,10339,12756,MR M,8,MR M
3,H,C25,10339,12756,MR M,9,MR M
4,H,C25,10339,12756,MR M,10,MR M
...,...,...,...,...,...,...,...
5379,F,5-4 G,10360,10684,MR,10,MR
5380,F,5-4 G,10360,10684,MR,11,MR
5381,F,5-4 G,10360,10684,MR,12,MR
5382,F,5-4 G,10360,10684,MR,13,MR


In [161]:
#Exportar

turnos_via.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/turnos_via_fms.csv',sep=';', index=False)

In [162]:
turnos_control.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/turnos_control_fms.csv',sep=';', index=False)

In [163]:
turnos_supervisor.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/turnos_supervisor_fms.csv',sep=';', index=False)

In [164]:
turnos_control_2026.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/turnos_control_fms_2026.csv',sep=';', index=False)

In [165]:
turnos_motoreg.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/turnos_motoregulador_fms.csv',sep=';', index=False)